# 2 · Loading, joining, saving — and xarray

How cubes get in and out of files: reading the metadata without the values, loading all or
part of a file, concatenating two rasters, writing a compressed NetCDF, and handing the
whole thing to `xarray`.

**Needs:** core dependencies only.

In [1]:
import importlib.util
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Environment noise, nothing to do with meteoraster: xarray probes every NetCDF/GRIB
# backend it can find (cfgrib needs the ecCodes binary), and this conda env's h5py and
# HDF5 builds differ by a patch release.
warnings.filterwarnings('ignore', message='(?s).*ecCodes.*')
warnings.filterwarnings('ignore', message='(?s).*h5py is running against HDF5.*')

# Report warnings without their absolute path, so the committed output does not carry
# one machine's directory layout. The warnings themselves stay visible.
def brief_warning(message, category, filename, lineno, line=None):
    return '%s:%d: %s: %s\n' % (Path(filename).name, lineno, category.__name__, message)


warnings.formatwarning = brief_warning

# Run straight from a clone: if `meteoraster` is not installed in this environment,
# fall back to the repository root one level up.
if importlib.util.find_spec('meteoraster') is None:
    sys.path.insert(0, str(Path('..').resolve()))

from meteoraster import MeteoRaster
from meteoraster import io as mrio

RESOURCES = Path('..') / 'tests' / 'resources'
FORECAST = RESOURCES / 'tethys_NOAA_GFS_0.25_2026.02.02.nct'

pd.set_option('display.width', 110)

## Look before you load

`io.read_header` opens the file, reads the coordinates and attributes, and closes it
again — **without reading a single data value**. On a multi-gigabyte archive this is the
difference between a question you can answer instantly and one that costs you your RAM.

In [2]:
header = mrio.read_header(FORECAST)

pd.Series({
    'variable': header.variable,
    'units': header.units,
    'shape': header.shape,
    'dtype': header.dtype,
    'productions': header.n_productions,
    'members': header.n_members,
    'leadtimes': header.n_leadtimes,
    'rows x cols': '%d x %d' % (header.rows, header.cols),
    'complete': header.complete,
    'leadtime format': header.leadtime_format,
    'latitudes flipped on read': header.flipped,
}).to_frame('header')

,header
variable,PRATE
units,mm/3h
shape,"(7, 1, 129, 29, 57)"
dtype,float32
productions,7
members,1
leadtimes,129
rows x cols,29 x 57
complete,False
leadtime format,pd.Timedelta


`complete` is a flag stored in the file by `save()`. `MeteoRaster.get_completeness` reads
just that one value:

In [3]:
MeteoRaster.get_completeness(FORECAST)

False

## Loading

`MeteoRaster.load` reads the whole cube. Reads go through `meteoraster.io`, which tries
`h5netcdf`, then `netcdf4`, then `scipy`, and pins the decoding choices that drift between
xarray releases — so a file written by a different install still comes back with the same
dtypes.

In [4]:
print('engines this install can use:', mrio.available_engines())

forecast = MeteoRaster.load(FORECAST, verbose=False)
print('loaded  ', forecast.data.shape, forecast.data.dtype,
      '  %.1f MB' % (forecast.data.nbytes / 1e6))

engines this install can use: ('h5netcdf', 'netcdf4', 'scipy')
loaded   (7, 1, 129, 29, 57) float32   6.0 MB


You can also read **part** of a file. `productions` and `leadtimes` accept indices, a
slice, or the coordinate values themselves; `dtype` casts while the cube is being filled
production by production, so it genuinely halves peak memory on a float64 file rather than
casting once everything is already resident.

In [5]:
first_day = MeteoRaster.load(FORECAST, leadtimes=slice(0, 9), dtype='float32', verbose=False)

print('partial ', first_day.data.shape, first_day.data.dtype,
      '  %.1f MB' % (first_day.data.nbytes / 1e6))
print('leadtimes kept:', [str(lead) for lead in pd.to_timedelta(first_day.leadtimes)])

partial  (7, 1, 9, 29, 57) float32   0.4 MB
leadtimes kept: ['0 days 00:00:00', '0 days 03:00:00', '0 days 06:00:00', '0 days 09:00:00', '0 days 12:00:00', '0 days 15:00:00', '0 days 18:00:00', '0 days 21:00:00', '1 days 00:00:00']


## Joining

`join` concatenates another raster onto this one along the production axis. To show it
working, split this file in two and put it back together.

> ⚠ `join` **modifies the object it is called on and returns `None`**. So do
> `a.join(b)`, not `c = a.join(b)`. The same is true of `trim()` and
> `resample_timestep()`.

In [6]:
dates = pd.to_datetime(forecast.production_datetime)
split_at = dates[3]

early = forecast.get_cropped(to_prod_date=dates[2])
late = forecast.get_cropped(from_prod_date=split_at)

print('early:', early.data.shape, [str(d)[:10] for d in pd.to_datetime(early.production_datetime)])
print('late :', late.data.shape, [str(d)[:10] for d in pd.to_datetime(late.production_datetime)])

early: (3, 1, 129, 29, 57) ['2026-02-02', '2026-02-03', '2026-02-04']
late : (4, 1, 129, 29, 57) ['2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08']


In [7]:
returned = early.join(late)

print('join() returned:', returned)
print('early is now   :', early.data.shape)
print('productions    :', [str(d)[:10] for d in pd.to_datetime(early.production_datetime)])
print('identical to the original cube:',
      np.array_equal(early.data, forecast.data, equal_nan=True))

join() returned: None
early is now   : (7, 1, 129, 29, 57)
productions    : ['2026-02-02', '2026-02-03', '2026-02-04', '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08']
identical to the original cube: True


What `join` does when the two rasters do not line up:

| Mismatch | Default behaviour | With `strickt=True` |
|---|---|---|
| different number of members | pads the smaller with NaN, warns | raises |
| different leadtimes | keeps the intersection, warns | raises |
| different lat/lon | raises | raises |
| overlapping production dates | aligns on the union, new data fills the NaN gaps | same |

`trim=True` drops leading and trailing all-NaN productions from both sides first.

## Saving

`save` writes compressed NetCDF through `h5netcdf`. The temporary file is staged **in the
destination directory** and swapped in with `os.replace`, so the swap is atomic and
nothing is routed through the system temp dir — which in a container is often a different
filesystem.

In [8]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'roundtrip.nct'

    first_day.save(path, complevel=4, complete=False)
    size_mb = path.stat().st_size / 1e6

    reloaded = MeteoRaster.load(path, verbose=False)

    print('written      : %.2f MB  (raw array is %.2f MB)'
          % (size_mb, first_day.data.nbytes / 1e6))
    print('values equal :', np.array_equal(first_day.data, reloaded.data, equal_nan=True))
    print('units        : %r -> %r' % (first_day.units, reloaded.units))
    print('variable     : %r -> %r' % (first_day.variable, reloaded.variable))
    print('leadtimes    :', [str(lead) for lead in pd.to_timedelta(reloaded.leadtimes)[:3]], '...')
    print('complete flag:', MeteoRaster.get_completeness(path))

written      : 0.13 MB  (raw array is 0.42 MB)


values equal : True
units        : 'mm/3h' -> 'mm/3h'
variable     : 'PRATE' -> 'PRATE'
leadtimes    : ['0 days 00:00:00', '0 days 03:00:00', '0 days 06:00:00'] ...
complete flag: False


## Exporting to xarray

`to_xarray()` hands you a labelled `xarray.DataArray` — the door to the whole xarray
ecosystem (`.sel`, `.groupby`, `.rolling`, dask, zarr, and so on).

Note how the spatial coordinates are laid out: `y` and `x` are the **index** dimensions
(plain integer positions), while `lat` and `lon` are **2-D non-index coordinates** over
`(y, x)`. That is what allows a curvilinear grid, where no single vector of latitudes
describes the rows.

In [9]:
array = first_day.to_xarray()
array

<xarray.DataArray 'PRATE' (production_datetime: 7, ensemble_member: 1,
                           leadtime: 9, y: 29, x: 57)> Size: 417kB
array([[[[[1.64159998e-01, 6.91199973e-02, 2.59200018e-02, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [1.72800004e-01, 8.64000022e-02, 1.90079987e-01, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [8.64000022e-02, 1.81440011e-01, 5.96159995e-01, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          ...,
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           8.63999967e-03, 8.63999967e-03, 8.63999967e-03]],

         [[0.00000000e+00, 8.63999967e-03, 8.63999967e-03, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           8.63999967e-03, 8.63999967e-03, 8.63999967e-03]],

         [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           4.31999983e-03, 4.31999983e-03, 4.31999983e-03],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           4.31999983e-03, 4.31999983e-03, 4.31999983e-03],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           8.63999967e-03, 8.63999967e-03, 8.63999967e-03],
          ...,
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [4.31999983e-03, 0.00000000e+00, 0.00000000e+00, ...,
           4.31999983e-03, 8.63999967e-03, 4.31999983e-03]]]]],
      shape=(7, 1, 9, 29, 57), dtype=float32)
Coordinates:
  * production_datetime  (production_datetime) datetime64[ns] 56B 2026-02-02 ...
  * ensemble_member      (ensemble_member) int64 8B 0
  * leadtime             (leadtime) timedelta64[ns] 72B 00:00:00 ... 1 days 0...
  * y                    (y) int64 232B 0 1 2 3 4 5 6 7 ... 22 23 24 25 26 27 28
  * x                    (x) int64 456B 0 1 2 3 4 5 6 7 ... 50 51 52 53 54 55 56
    lat                  (y, x) float64 13kB 54.0 54.0 54.0 ... 47.0 47.0 47.0
    lon                  (y, x) float64 13kB -2.0 -1.75 -1.5 ... 11.5 11.75 12.0
    leadtime_format      <U12 48B 'pd.Timedelta'
Attributes:
    units:     mm/3h
    variable:  PRATE
    saved:     Saved by MeteoRaster v3.0 @ 2026-08-14T16:19:50.156794

In [10]:
print('dims  :', array.dims)
print('coords:', list(array.coords))
print('attrs :', {k: v for k, v in array.attrs.items() if k != 'saved'})

dims  : ('production_datetime', 'ensemble_member', 'leadtime', 'y', 'x')
coords: ['production_datetime', 'ensemble_member', 'leadtime', 'lat', 'lon', 'y', 'x', 'leadtime_format']
attrs : {'units': 'mm/3h', 'variable': 'PRATE'}


Selecting by label, and reducing over the axes you do not care about:

In [11]:
one_map = array.sel(ensemble_member=0).isel(production_datetime=0, leadtime=4)
print('one map:', one_map.shape, '| valid at',
      pd.to_datetime(first_day.production_datetime)[0] + pd.to_timedelta(first_day.leadtimes)[4])

mean_field = array.mean(dim=['production_datetime', 'ensemble_member', 'leadtime'])
print('mean field:', mean_field.shape, '| mean value %.4f %s'
      % (float(mean_field.mean()), first_day.units))

one map: (29, 57) | valid at 2026-02-02 12:00:00
mean field: (29, 57) | mean value 0.2980 mm/3h


From there, `array.to_netcdf(...)` writes a conventional NetCDF (readable by anything),
as opposed to `MeteoRaster.save`, which writes the layout `MeteoRaster.load` expects.

## Files too large to hold

`load()` puts the whole cube in memory. When that is not an option, read the header first
and then stream one production at a time — peak memory is one block, whatever the file
size. `reduce_members` collapses the ensemble axis as it goes.

In [12]:
totals = []
for production, block in mrio.iter_productions(FORECAST, reduce_members='mean'):
    totals.append({
        'production': pd.Timestamp(production),
        'block shape': block.shape,                     # [leadtime, y, x] once members are reduced
        'block MB': round(block.nbytes / 1e6, 2),
        'domain total (%s)' % header.units: round(float(np.nansum(block)), 1),
    })

pd.DataFrame(totals).set_index('production')

,block shape,block MB,domain total (mm/3h)
production,,,
2026-02-02,"(129, 29, 57)",0.85,81364.5
2026-02-03,"(129, 29, 57)",0.85,93639.2
2026-02-04,"(129, 29, 57)",0.85,86311.0
2026-02-05,"(129, 29, 57)",0.85,87029.7
2026-02-06,"(129, 29, 57)",0.85,118062.2
2026-02-07,"(129, 29, 57)",0.85,0.0
2026-02-08,"(129, 29, 57)",0.85,94088.5


One block is a seventh of the cube here; on a real archive the ratio is what matters.
`io.read_cube` sits in between — it fills a whole or partial cube the same way, production
by production.

Note the `0.0` on 2026-02-07. That is `np.nansum` of an all-NaN block, not a dry day — it
is the missing production from notebook 1. Streaming gives you the raw blocks, so
completeness is still yours to check.

## Reading GRIB

Converted Tethys output needs no reader — `MeteoRaster.load` reads it directly, as above.
For raw ERA5-Land GRIB from the Copernicus CDS, `meteoraster.utils` has two readers that
return a ready `MeteoRaster`:

```python
from meteoraster.utils import readERA5Land_monthly, read_ERA5Land_hourly

# monthly means: tp comes back as mm/month, t2m as degC
monthly = readERA5Land_monthly('era5land_monthly.grib', 'tp')

# hourly: accumulations are de-accumulated to per-hour increments (mm/hr).
# A CDS request omits the last hour of its final day - it is delivered with the
# next period, so pass that file in to fill the gap.
hourly = read_ERA5Land_hourly('era5land_2025_11.grib', 'tp',
                              file_next_year='era5land_2025_12.grib')
```

These cells are **not executed here**: `cfgrib` needs the ecCodes binary, which on Windows
lives in `<env>/Library/bin` and is found only through an *activated* conda environment.
Calling the environment's `python.exe` directly raises
`RuntimeError: Cannot find the ecCodes library` even though the package is installed.
`tests/test_utils_readers.py` exercises both readers against the GRIB fixtures.

---
**Next:** [03 · Cropping](03_cropping.ipynb) — cutting the cube down in space, in
production time and in leadtime.